# FIUBench Forget5 재평가 — fast tokenizer + 원본 generation 경계

이 노트북은 기존 **GA 또는 PO 최종 `checkpoint.pt`**를 다시 학습하지 않고 평가합니다.

해결하는 문제:

- `tokenizers==0.15.1`이 모델의 `tokenizer.json`을 읽지 못한 문제
- `use_fast=False` 우회 뒤 `labels != -100` 경계 패치가 `<|assistant|>`를 제거한 문제
- 평가 중 checkpoint 경로 혼동과 중간 `step_*` checkpoint 오선택
- Kaggle T4에서 Flash Attention을 사용할 수 없는 문제

평가 환경은 예전 성공 환경과 맞춘 `transformers 4.46.3`, `tokenizers 0.20.3`,
`peft 0.13.2`, `accelerate 0.34.2`입니다. `LlamaTokenizerFast`와 공개 원본의
문자열 answer-tag 분리가 실제로 호환되는지 사전 검증에 통과해야 평가가 시작됩니다.

## 시작 전

1. Kaggle에서 GPU T4를 켭니다.
2. GA 또는 PO 학습 Notebook의 저장된 Version Output을 **Add Input**으로 하나만 연결합니다.
3. 셀을 순서대로 실행합니다.
4. 설치 셀 다음에 커널 재시작 셀을 한 번 실행합니다.

평가는 두 부분으로 나뉩니다.

- **Core**: Forget5/Retain5의 ROUGE, EM, loss, Truth Ratio, Min-K 계열
- **APE**: Forget5 paraphrased question 1,200개. 선택 실행

`GPT-Eval`, `MME`, `POPE`, 완전한 MIA/KS 집계는 이 노트북의 범위가 아닙니다.


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working
if [ ! -d FIUBench_kaggle ]; then
  git clone https://github.com/npngjjm/FIUBench_kaggle.git
fi

python -m pip uninstall -y torchao || true
python -m pip install --no-cache-dir --upgrade \
  'transformers==4.46.3' \
  'tokenizers==0.20.3' \
  'peft==0.13.2' \
  'accelerate==0.34.2' \
  'huggingface_hub==0.36.2' \
  'numpy==2.1.3' \
  hydra-core omegaconf datasets sentencepiece protobuf scipy pillow \
  pyyaml requests tqdm rouge_score scikit-learn

echo 'INSTALL_COMPLETE — 다음 셀에서 Python 커널을 재시작하세요.'


설치가 끝난 뒤 바로 아래 셀을 한 번 실행하세요. Python 프로세스만 재시작하며,
`/kaggle/working`의 clone과 설치된 패키지는 유지됩니다. 연결이 잠시 끊겼다가 자동 복구됩니다.
그 후에는 이 셀을 다시 실행하지 말고 다음 셀부터 진행하세요.


In [ ]:
import os
os._exit(0)


아래 `METHOD`를 연결한 checkpoint에 맞게 `"GA"` 또는 `"PO"`로 지정하세요.


In [ ]:
METHOD = "GA"  # "GA" 또는 "PO"

assert METHOD in {"GA", "PO"}
print("평가 방법:", METHOD)


In [ ]:
import os
import shutil
import sys
from importlib.metadata import version
from pathlib import Path

import torch

repo = Path("/kaggle/working/FIUBench_kaggle")
assert repo.is_dir(), "저장소가 없습니다. 첫 설치/clone 셀을 다시 실행하세요."
os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

expected = {
    "transformers": "4.46.3",
    "tokenizers": "0.20.3",
    "peft": "0.13.2",
    "accelerate": "0.34.2",
    "huggingface-hub": "0.36.2",
    "numpy": "2.1.3",
}

print("=== 라이브러리 ===")
for package, wanted in expected.items():
    installed = version(package)
    print(package, installed)
    assert installed == wanted, (package, installed, wanted)

assert torch.cuda.is_available(), "Kaggle에서 GPU를 켜세요."
print("torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

# Add Input에서 최종 checkpoint만 선택합니다. step_* 중간 checkpoint는 제외합니다.
candidates = [
    p for p in Path("/kaggle/input").rglob("checkpoint.pt")
    if not any(part.startswith("step_") for part in p.parts)
]

print("최종 checkpoint 후보:", *map(str, candidates), sep="\n")
assert len(candidates) == 1, (
    "최종 checkpoint 후보가 정확히 하나여야 합니다. "
    "GA 또는 PO 학습 Output 하나만 Add Input으로 연결하세요."
)

run_name = f"{METHOD.lower()}_forget5_fast_eval"
run_dir = Path("/kaggle/working") / run_name
checkpoint = run_dir / "checkpoint.pt"
run_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(candidates[0], checkpoint)

state = torch.load(checkpoint, map_location="cpu", weights_only=True)
assert isinstance(state, dict) and state
print("checkpoint:", checkpoint)
print("checkpoint 텐서 수:", len(state))
print("checkpoint 크기 MiB:", round(checkpoint.stat().st_size / 1024**2, 1))
del state


In [ ]:
# fast tokenizer가 원본 evaluate_util.py의 answer boundary와 호환되는지 확인합니다.
from transformers import AutoTokenizer
from utils import get_model_identifiers_from_yaml

model_path = "gray311/vlm_unlearning_ft_llava_phi_3_mini"
model_family = "llava-phi"

tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)
assert tokenizer.is_fast

model_cfg = get_model_identifiers_from_yaml(model_family)
question_tag = model_cfg["question_start_tag"]
answer_tag = model_cfg["answer_tag"]
original_boundary = answer_tag.replace("\n", "")

conversation = (
    question_tag
    + "<image>\n"
    + "What is the full name of the person in the image?"
    + answer_tag
    + "The full name is test person."
)
ids = tokenizer(conversation, return_tensors="pt")["input_ids"]
decoded = tokenizer.batch_decode(ids, skip_special_tokens=False)[0]

print("tokenizer class:", tokenizer.__class__.__name__)
print("is_fast:", tokenizer.is_fast)
print("원본 boundary:", repr(original_boundary))
print("decoded:", repr(decoded))
print("boundary 발견:", original_boundary in decoded)

assert tokenizer.__class__.__name__ == "LlamaTokenizerFast"
assert original_boundary in decoded, (
    "Fast tokenizer에서도 공개 원본의 answer boundary를 찾지 못했습니다. "
    "이 상태에서는 평가를 실행하지 마세요."
)

prompt = decoded.split(original_boundary, 1)[0].strip(" ") + original_boundary
prompt_ids = tokenizer(prompt, add_special_tokens=True)["input_ids"]
assistant_id = tokenizer.convert_tokens_to_ids("<|assistant|>")
assert prompt_ids[-1] == assistant_id, (
    "생성 prompt가 <|assistant|> 토큰으로 끝나지 않습니다."
)
print("생성 prompt 마지막 토큰:", prompt_ids[-1], "(<|assistant|>)")
print("FAST_TOKENIZER_AND_BOUNDARY_OK")


In [ ]:
# forget5 + retain5 평가에 필요한 40개 이미지만 공개 archive에서 복원합니다.
import json
import shutil
import zipfile
from collections import defaultdict
from pathlib import Path

from huggingface_hub import HfApi, hf_hub_download
from PIL import Image

records = [
    json.loads(line)
    for line in (repo / "dataset/full.json").read_text().splitlines()
    if line.strip()
][:400]
split_info = json.loads((repo / "dataset/split.json").read_text())
selected_ids = set(split_info["forget5"] + split_info["retain5"])
needed = {
    row["image_path"]
    for row in records
    if row["unique_id"] in selected_ids
}
assert len(needed) == 40

revision = HfApi().dataset_info("gray311/FIUBench").sha
archive = hf_hub_download(
    repo_id="gray311/FIUBench",
    filename="SFHQ.zip",
    repo_type="dataset",
    revision=revision,
    local_dir="/kaggle/working/fiu_download",
)

with zipfile.ZipFile(archive) as zipped:
    entries = defaultdict(list)
    for member in zipped.infolist():
        path = Path(member.filename)
        if (
            not member.is_dir()
            and "__MACOSX" not in path.parts
            and not path.name.startswith("._")
        ):
            entries[path.name].append(member)

    for image_path in needed:
        destination = (repo / image_path).resolve()
        assert destination.is_relative_to((repo / "dataset").resolve())
        matches = entries[destination.name]
        assert len(matches) == 1, (destination.name, len(matches))
        destination.parent.mkdir(parents=True, exist_ok=True)
        with zipped.open(matches[0]) as source, destination.open("wb") as target:
            shutil.copyfileobj(source, target)
        with Image.open(destination) as image:
            image.verify()

assert all((repo / path).is_file() for path in needed)
(run_dir / "fiu_eval_data_revision.txt").write_text(revision)
print("검증한 이미지:", len(needed))
print("dataset revision:", revision)


In [ ]:
# Kaggle 실행 복사본에 환경 호환 패치만 적용합니다.
# run_generation의 원본 문자열 boundary 로직은 변경하지 않습니다.
import ast
from pathlib import Path

source_path = repo / "evaluate_util.py"
source = source_path.read_text()

# evaluate_util.py에서는 이 함수가 실제로 사용되지 않습니다.
# forget.py 전체 import가 불필요하게 deepspeed를 요구하므로 제거합니다.
source = source.replace("from forget import find_all_linear_names\n", "")

api_import = '''from data_generation.api import (
    GeminiEvaluator, 
    GPTEvaluator, 
    system_message, 
    user_message, 
    jobs
)'''
api_shim = '''class GPTEvaluator:
    def __init__(self, *args, **kwargs): pass
    def generate_answer(self, *args, **kwargs):
        raise RuntimeError("GPT-Eval은 이 노트북에 포함되지 않습니다.")
GeminiEvaluator = GPTEvaluator
system_message = user_message = jobs = None
'''

assert api_import in source or api_shim in source
source = source.replace(api_import, api_shim)

source = source.replace(
    "AutoTokenizer.from_pretrained(cfg.model_path)",
    "AutoTokenizer.from_pretrained(cfg.model_path, use_fast=True)",
)
source = source.replace(
    'attn_implementation="flash_attention_2"',
    'attn_implementation="sdpa"',
)

old_load = "model.load_state_dict(torch.load(cfg.LoRA.lora_path), strict=False)"
safe_load = '''load_result = model.load_state_dict(
                torch.load(cfg.LoRA.lora_path, map_location="cpu", weights_only=True),
                strict=False,
            )
            if load_result.unexpected_keys:
                raise RuntimeError(
                    f"Checkpoint unexpected keys: {load_result.unexpected_keys[:10]}"
                )'''
assert old_load in source or safe_load in source
source = source.replace(old_load, safe_load)

old_cuda = "    model.half().cuda()"
eval_cuda = "    model.half().cuda()\n    model.eval()"
assert old_cuda in source or eval_cuda in source
source = source.replace(old_cuda, eval_cuda)

# 잘못된 labels 경계 패치가 섞이지 않았는지 확인합니다.
required_original = '''    input_strings = tokenizer.batch_decode(input_ids)

    model_config = get_model_identifiers_from_yaml(cfg.model_family)
    question_start_tag = model_config['question_start_tag']
    answer_tag = model_config['answer_tag']
    answer_tag = answer_tag.replace("\\n", "") 

    ground_truth = [s.split(answer_tag)[1].strip(" ") for s in input_strings]
    input_strings = [s.split(answer_tag)[0].strip(" ") for s in input_strings]
    input_strings = [s + answer_tag for s in input_strings]
'''
assert required_original in source, (
    "공개 원본 run_generation 블록이 아닙니다. "
    "저장소를 새로 clone한 뒤 다시 실행하세요."
)
assert 'labels = batch["labels"]\n    input_strings = []' not in source

ast.parse(source)
backup = source_path.with_suffix(".py.before_fast_eval")
if not backup.exists():
    backup.write_text(source_path.read_text())
source_path.write_text(source)

print("FAST_TOKENIZER=True")
print("GENERATION_BOUNDARY=UPSTREAM_STRING_SPLIT")
print("ATTENTION=SDPA")
print("런타임 평가 패치 검증 완료")


In [ ]:
# Core와 APE를 분리해 중간 결과를 보존하고 선택 실행할 수 있게 합니다.
import yaml

base = yaml.safe_load((repo / "config/eval.yaml").read_text())
base["model_path"] = "gray311/vlm_unlearning_ft_llava_phi_3_mini"
base["LoRA"]["lora_path"] = str(checkpoint)
base["workers"] = 0
base["batch_size"] = 1
base["perturb_batch_size"] = 1
base["overwrite"] = False
base["save_dir"] = str(run_dir / "eval_results")

core = dict(base)
core["data_path"] = ["./dataset/full.json"] * 2
core["split_list"] = ["forget5", "retain5"]
core["question_key"] = ["question"] * 2
core["robust_question_key"] = ["question"] * 2
core["answer_key"] = ["answer"] * 2
core["base_answer_key"] = ["paraphrased_answer"] * 2
core["perturbed_answer_key"] = ["perturbed_answer"] * 2
core["eval_task"] = ["eval_forget_log", "eval_retain_log"]
core["robust_eval"] = [["exact_match", "mink"], ["rouge"]]

ape = dict(base)
ape["data_path"] = ["./dataset/full.json"]
ape["split_list"] = ["forget5"]
ape["question_key"] = ["question"]
ape["robust_question_key"] = ["paraphrased_question"]
ape["answer_key"] = ["answer"]
ape["base_answer_key"] = ["paraphrased_answer"]
ape["perturbed_answer_key"] = ["perturbed_answer"]
ape["eval_task"] = ["eval_forget_ape"]
ape["robust_eval"] = [["ape"]]

core_name = f"eval_{METHOD.lower()}_forget5_fast_core"
ape_name = f"eval_{METHOD.lower()}_forget5_fast_ape"
core_path = repo / "config" / f"{core_name}.yaml"
ape_path = repo / "config" / f"{ape_name}.yaml"
core_path.write_text(yaml.safe_dump(core, sort_keys=False, allow_unicode=True))
ape_path.write_text(yaml.safe_dump(ape, sort_keys=False, allow_unicode=True))

print("Core config:", core_path)
print("Core tasks:", core["eval_task"])
print("APE config:", ape_path)
print("APE task:", ape["eval_task"])
print("checkpoint:", checkpoint)


## 1단계: Core 평가

Forget5 400개와 Retain5 400개를 평가합니다. 이전 결과와 직접 비교할 핵심 단계입니다.
Kaggle T4 한 장 기준으로 대략 **1~1.5시간**을 예상하지만 환경에 따라 달라질 수 있습니다.


In [ ]:
import os
import subprocess

env = os.environ.copy()
env.update({
    "HF_HOME": "/kaggle/working/hf_cache",
    "TOKENIZERS_PARALLELISM": "false",
    "PYTHONUNBUFFERED": "1",
    "HYDRA_FULL_ERROR": "1",
})
core_log = run_dir / "core_console.log"
command = [
    sys.executable,
    "evaluate_util.py",
    "--config-name",
    core_name,
]
with core_log.open("w") as log:
    process = subprocess.Popen(
        command,
        cwd=repo,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
        log.write(line)
    return_code = process.wait()
assert return_code == 0, f"Core 평가 실패: exit {return_code}"
print("CORE_EVAL_PROCESS_EXIT_0")


In [ ]:
# Core 결과를 즉시 요약합니다.
import json
import math
import statistics

result_dir = run_dir / "eval_results"
forget_file = result_dir / "forget5_eval_forget_log.json"
retain_file = result_dir / "retain5_eval_retain_log.json"
assert forget_file.is_file(), forget_file
assert retain_file.is_file(), retain_file

forget = json.loads(forget_file.read_text())
retain = json.loads(retain_file.read_text())

def values(obj):
    if isinstance(obj, dict):
        obj = list(obj.values())
    if not isinstance(obj, list):
        return []
    return [float(x) for x in obj if isinstance(x, (int, float))]

def mean_metric(data, key):
    numeric = values(data.get(key))
    return statistics.fmean(numeric) if numeric else None

print("=" * 68)
print(f"FIUBench {METHOD} Forget5 — FAST tokenizer Core 결과")
print("=" * 68)
for title, data in [("Forget", forget), ("Retain", retain)]:
    print(f"\n[{title}]")
    for key in [
        "exact_match",
        "rouge1_recall",
        "rougeL_recall",
        "truth_ratio",
        "mink",
        "mink++",
    ]:
        value = mean_metric(data, key)
        if value is not None:
            print(f"{key:20s}: {value:.6f} ({value * 100:.4f}%)")
print("\n표본 수 — Forget generated:", len(forget.get("generated_text", {})))
print("표본 수 — Retain generated:", len(retain.get("generated_text", {})))
print("결과 폴더:", result_dir)


## 2단계: APE 평가 — 선택 실행

논문의 APE와 비교하려면 아래 셀도 실행하세요. Forget5의 paraphrased question
1,200개를 생성하므로 추가로 대략 **1~1.5시간**이 걸릴 수 있습니다.

Core 결과만 먼저 확인하려면 여기서 멈춰도 됩니다. 이후 같은 저장 output을 연결하고
위 셀들을 실행하면 `overwrite: false` 때문에 완료된 Core JSON은 다시 계산하지 않습니다.


In [ ]:
ape_log = run_dir / "ape_console.log"
command = [
    sys.executable,
    "evaluate_util.py",
    "--config-name",
    ape_name,
]
with ape_log.open("w") as log:
    process = subprocess.Popen(
        command,
        cwd=repo,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
        log.write(line)
    return_code = process.wait()
assert return_code == 0, f"APE 평가 실패: exit {return_code}"
print("APE_EVAL_PROCESS_EXIT_0")


In [ ]:
ape_file = result_dir / "forget5_eval_forget_ape.json"
assert ape_file.is_file(), ape_file
ape_result = json.loads(ape_file.read_text())
ape_scores = values(ape_result.get("exact_match"))
assert len(ape_scores) == 1200, len(ape_scores)
print("APE samples:", len(ape_scores))
print("APE mean:", statistics.fmean(ape_scores))
print("APE percent:", f"{statistics.fmean(ape_scores) * 100:.4f}%")
print("모든 공개 core + APE 결과 생성 완료")


## 보존

완료 후 **Save Version → Quick Save**로 `/kaggle/working` output을 보존하세요.
보존해야 할 폴더는 Method에 따라 다음 중 하나입니다.

- `/kaggle/working/ga_forget5_fast_eval`
- `/kaggle/working/po_forget5_fast_eval`

여기에는 복사된 최종 checkpoint, `eval_results`, console log, dataset revision이 들어 있습니다.
